In [1]:
import os
import requests
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

print("✅ Imports OK")

✅ Imports OK


In [2]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

vector_store = FAISS.load_local(
    "vector_store",
    embeddings,
    allow_dangerous_deserialization=True
)

print("✅ Vector store loaded")
print(f"Vectors: {vector_store.index.ntotal}")

c:\Users\ahmed\anaconda3\envs\ml_env\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.16) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✅ Vector store loaded
Vectors: 21


In [3]:
def retrieve(question, k=5):
    docs_and_scores = vector_store.similarity_search_with_score(question, k=k)
    return [doc for doc, score in docs_and_scores]

# تست
docs = retrieve("What services does CrocoIT offer?")
print(f"✅ Retrieved {len(docs)} chunks")
print(docs[0].page_content[:200])

✅ Retrieved 5 chunks
CrocoIT is one of the fastest growing software houses
Start your business with CrocoIT
We are ready to design and develop a custom e-commerce platform (online Store ), web application, commerce mobile


In [8]:
import getpass

os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter Gemini API Key: ")

def call_gemini(prompt):
    api_key = os.environ["GOOGLE_API_KEY"]
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={api_key}"
    body = {
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"temperature": 0.3, "maxOutputTokens": 512}
    }
    response = requests.post(url, json=body)
    data = response.json()
    if response.status_code == 200:
        return data["candidates"][0]["content"]["parts"][0]["text"]
    else:
        return f"❌ Error: {data['error']['message']}"

print("✅ Gemini ready")

✅ Gemini ready


In [9]:
def ask(question):
    print(f"\n❓ Question: {question}")
    print("-" * 60)

    docs = retrieve(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are a helpful assistant for CrocoIT, a software company.
Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't have enough information about that."
Be concise, professional, and friendly.

Context:
{context}

Question:
{question}

Answer:
"""
    answer = call_gemini(prompt)

    print(f"💬 Answer:\n{answer}")
    print("\n📄 Sources:")
    for doc in docs:
        print(f"  - {doc.metadata['url']}")
    print("=" * 60)

print("✅ ask() ready")

✅ ask() ready


In [11]:
ask("What products does CrocoIT have?")


❓ Question: What products does CrocoIT have?
------------------------------------------------------------
💬 Answer:
CrocoIT offers custom e-commerce platforms (online stores), web applications, commerce mobile apps, mobile applications for Android and iOS, web and mobile apps, and ERP systems. They also offer HR Mobic and Agile management systems.

📄 Sources:
  - https://crocoit.com/about-us/
  - https://crocoit.com/about-us/
  - https://crocoit.com
  - https://crocoit.com/our-products/
  - https://crocoit.com
